[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-biology/AdvancesMSDataProcessing/blob/master/StatisticalValidationModelRobustness/Metanorm_figures.ipynb)


# Metanorm — figure-by-figure walkthrough

**Paper:** *Robust metabolomics data normalization across scales and experimental designs*
(Vynck, Vangeenderhuysen, De Paepe, Nawrot, Plekhova, Vanhaecke),
bioRxiv [10.1101/2025.09.30.679445v2](https://www.biorxiv.org/content/10.1101/2025.09.30.679445v2.full)

**Code:** [UGent-LIMET/Metanorm_publication/Rcode](https://github.com/UGent-LIMET/Metanorm_publication/tree/main/Rcode) ·
**Package:** [UGent-LIMET/Metanorm](https://github.com/UGent-LIMET/Metanorm)

This notebook pairs each main-text figure with an explanation of what it shows, how to read it, and
the code excerpt that produces it. Explanations are written here; they are not the published captions.

---

### Important: figure numbers in the repo do not match the v2 preprint

The scripts write into `Figures/ForPublication/FigureN/` folders that follow an earlier numbering.
Use this mapping when hunting for a panel:

| Preprint v2 | Content | Repo output folder | Script |
|---|---|---|---|
| Fig 1a | BioHEART example metabolite, 8 methods | `Figure1/prepost-*.pdf` | `Dataset-BioHeart.R` |
| Fig 1b | MSPE vs. number of outliers (in silico) | `aaForPublication/Figure1/Figure1B.pdf` | `FigureMSPE.R` (+ `simStudy-*.R`) |
| Fig 2 | ENVIRONAGE weighting / QCcheck demo | *(no dedicated script)* | — |
| Fig 3 | BioHEART: intensities, DAM count, Venn, PCA, heatmap | `Figure3/`, `Figure1/venn.pdf` | `Dataset-BioHeart.R` |
| Fig 4 | Replicate-run MAD (BioHEART) | *(no script found)* | — |
| Fig 5 | ENVIRONAGE: same panel set | `Figure4/` | `Dataset-ENVIRONAGE.R` |
| Fig 6 | Compute time vs. cores | `Figure5/Figure5a-b.pdf` | `FigureTime.R` (+ `methodComparisonTime*.R`) |
| Suppl. FAME | Same panel set, untargeted | `FigureSupplFAME/` | `dataset-FAME.R` |

### Before running anything

None of these cells run end-to-end out of the box: the scripts expect the three experimental datasets on
disk under `Data/`, plus SERRF output downloaded from its web interface, and hard-code output paths such
as `Figures/ForPublication/...`. Create those folders, or edit the `ggsave()` paths, before executing.
Data sources are listed in the last section.

## 0. Setup

Shared libraries and helper functions. Every dataset script opens with the same block: a compact
ggplot theme, a pre/post-normalization sequence plotter, a PCA score plotter, and a Venn wrapper.
`metanorm` is the paper's package; `TIGERr` provides the competing ensemble method.

In [ ]:
install.packages(c("ggplot2", "cowplot", "dplyr", "reshape2", "VennDiagram",
                   "extrafont", "readxl", "fANCOVA", "mgcv", "TIGERr"))
remotes::install_github("UGent-LIMET/Metanorm")

In [ ]:
# ---- Dataset-BioHeart.R, lines 6-13 ----
library(metanorm)
library(TIGERr)
library(ggplot2)
library(cowplot)
library(dplyr)
library(reshape2)
library(VennDiagram)
library(extrafont)

In [ ]:
# ---- Dataset-BioHeart.R, lines 15-42 ----
theme_custom <- function(base_size = 11, base_family = "Arial") {
  theme_classic(base_size = base_size, base_family = base_family) +
    theme(
      # grid and axes
      axis.line      = element_line(color = "black", linewidth = 0.5),
      axis.ticks     = element_line(color = "black", linewidth = 0.5),
      axis.ticks.length = unit(2, "pt"),
      axis.text      = element_text(color = "black"),
      
      # legend
      legend.key     = element_blank(),
      legend.title   = element_text(size = base_size),
      legend.text    = element_text(size = base_size),
      
      # strip for facet
      strip.background = element_blank(),
      strip.text       = element_text(face = "bold", size = base_size),
      
      # panel
      panel.border   = element_blank(),
      panel.grid     = element_blank(),
      
      # text
      text           = element_text(family = base_family, size = base_size),
      plot.title     = element_text(hjust = 0, face = "bold", size = base_size + 1),
      plot.margin    = margin(4, 4, 4, 4, "pt")
    )
}

In [ ]:
# ---- Dataset-BioHeart.R, lines 44-118 ----
plotSequence <- function(raw, datn, order = NULL, batch, type){
  if(is.null(order)){order <- 1:length(datn)}
  
  # make data frame for plotting
  rawdf <- data.frame(x=order,
                      y=raw,
                      batch=batch,
                      type=type)
  rawdf$batch <- as.factor(rawdf$batch)
  
  # plot data before normalization
  preNorm <- ggplot(rawdf, aes(x = x,
                               y = y,
                               shape = batch,
                               col = type))+
    geom_point()+
    scale_shape_manual(values = seq(1:length(levels(rawdf$batch))))+
    theme_minimal()+
    xlab("injection order")+
    ylab("intensity")+
    theme(legend.text = element_text(size = 5),
          legend.title = element_text(size = 5),
          legend.key.height = unit(5, "points"),
          legend.spacing.y = unit(0, "cm"))
  
  
  # make data from for plotting
  datn <- data.frame(y=datn, x=order, batch, type)
  datn$batch <- as.factor(datn$batch)
  
  # plot data after normalization
  postNorm <- ggplot(datn, aes(x = .data$x,
                               y = .data$y,
                               shape = batch,
                               col = type))+
    geom_point()+
    scale_shape_manual(values = seq(1:length(levels(datn$batch))))+
    theme_minimal()+
    xlab("injection order")+
    ylab("normalized ntensity")+
    theme(legend.text = element_text(size = 5),
          legend.title = element_text(size = 5),
          legend.key.height = unit(5, "points"),
          legend.spacing.y = unit(0, "cm"))
  
  # return combined plot pre- vs post normalization
  p <- plot_grid(preNorm, postNorm, nrow = 2)
}

plotPCA2 <- function (x, type) 
{
  pca.result <- prcomp(t(x), scale = TRUE)
  pca.data <- data.frame(PC1 = pca.result$x[, 1], PC2 = pca.result$x[, 
                                                                     2], batch = type)
  pca.data$batch <- factor(pca.data$batch, levels = paste0("batch", 1:15))
  plot <- ggplot(pca.data, aes(x = .data$PC1, y = .data$PC2, 
                               color = .data$batch, shape = .data$batch)) + geom_point(size = 0.7, alpha = 0.7) + 
    labs(title = element_blank(), x = "PC1", y = "PC2") + 
    
    scale_color_manual(values = colors)+
    #scale_color_grey() +
    theme_custom() + 
    
    scale_shape_manual(values = seq(1:length(levels(pca.data$batch))))+
    theme(legend.text = element_text(margin = margin(t = 1, b = 1)),
          legend.key.height    = unit(7, "pt"),)
  
  return(plot)
}

display_venn <- function(x, ...){
  grid.newpage()
  venn_object <- venn.diagram(x, filename = NULL, ...)
  grid.draw(venn_object)
}

---

## Figure 1 — how badly do outliers hurt each method?

This is the paper's opening argument, in two halves.

**Panel (a)** takes one metabolite from the BioHEART plasma study and draws its intensity against
injection order eight times: unnormalized, then after QC-RLSC, QC-RSC, SERRF, TIGER, rLOESS, rGAM and
tGAM. Dots are QC injections, triangles are biological samples, and colour encodes the batch (15 of
them). The point is visual: in batches that contain an outlying intensity, the non-robust fits get
dragged toward the outlier, so the corrected samples around it are pushed off the common level and the
batches no longer line up. The three robust methods leave that neighbourhood alone.

**Panel (b)** makes the same claim causally, on simulated data where the truth is known. The x-axis is
the number of injected outliers (0 to 4); the y-axis is median squared prediction error as a percentage
of the QC-RLSC / zero-outlier reference, so 100% is the baseline. Solid lines use 1 QC per 10 samples,
dashed lines 2 per 10. Two things to read off: where each line starts (accuracy with clean data) and how
steeply it rises (sensitivity to outliers). The robust curves stay comparatively flat.

### 1a — BioHEART data, normalization by every method

Load the filtered BioHEART matrix, derive the QC/sample vector and the batch vector from its header
rows, log-transform, then run all five `metanorm` models. `QCcheck = TRUE` turns on the QC-versus-sample
representativeness test for the GAM methods; `QConly = TRUE` restricts QC-RLSC and QC-RSC to QC
injections, as in their original definitions.

In [ ]:
# ---- Dataset-BioHeart.R, lines 126-148 ----
# data as obtained from GitHub: sydneyBioX/BioHEART_metabolomics
metab <- read.csv("BioHEART_metabolomics_filtered.csv", header = TRUE)

# recode QC vs non QC
metanorm.qc <- unname(unlist(metab[2, 2:ncol(metab)]))
metanorm.qc[metanorm.qc == "QC"] <- "QC"
metanorm.qc[metanorm.qc != "QC"] <- "sample"

# extract batch information
batch <- paste0("batch", unlist(metab[3, 2:ncol(metab)]))

# get intensity data only, and log transform
rawdata <- log(read.csv("~/Downloads/BioHEART_metabolomics_filtered.csv", header = TRUE, skip = 3)[,-1])

# perform normalization
GAMnorm <- metanorm(rawdata, type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/BioHeart-batchw-metanorm-2/")
rGAMnorm <- metanorm(rawdata, model="rGAM", type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/BioHeart-batchwrgam-metanorm-2/")
QCRLSCLOOCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "LOOCV", plotdir = "Figures/BioHeart--metanorm-QCRLSC-LOOCV/")
QCRLSCGCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "GCV", plotdir = "Figures/BioHeart--metanorm-QCRLSC-GCV/")
QCRSCLOOCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "LOOCV", plotdir = "Figures/BioHeart--metanorm-QCRSC-LOOCV/")
QCRSCGCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "GCV", plotdir = "Figures/BioHeart--metanorm-QCRSC-GCV/")
rLOESSLOOCVnorm <- metanorm(rawdata, type = metanorm.qc, batch = batch, model = "rLOESS", cv = "LOOCV", plotdir = "Figures/BioHeart--metanorm-rLOESS-LOOCV/")
rLOESSGCVnorm <- metanorm(rawdata, type = metanorm.qc, batch = batch, model = "rLOESS", cv = "GCV", plotdir = "Figures/BioHeart--metanorm-rLOESS-GCV/")

TIGER needs a transposed frame with explicit batch / type / order columns, and its own train/test
split where the training set is the QCs. SERRF has no R package: the authors exported a CSV, ran it
through the SERRF web interface, and read the result back. Two runs (544, 1178) are dropped on the
SERRF side, which is why its indices are subset below.

In [ ]:
# ---- Dataset-BioHeart.R, lines 150-176 ----
# TIGERr through its own R package
# custom data format
tigerdat <- data.frame(t(rawdata),
                       batch = batch,
                       type = metanorm.qc,
                       order = 1:ncol(rawdata))
test_samples <- tigerdat
train_samples <- tigerdat[tigerdat$type == "QC",]
train_samples$sampleID <- "train"
test_samples$sampleID  <- "test"

# normalize
tigernorm <- run_TIGER(test_samples,
                       train_samples,
                       col_sampleID = "sampleID",
                       col_sampleType = "type",
                       col_batchID = "batch",
                       col_order = "order")



plotdir = "Figures/BioHeart--metanorm-TIGER/"
for(i in 1:ncol(tigernorm)){
  pdf(paste0(plotdir, "/", i, ".pdf"), width = 8, height = 5)
  print(plotSequence(unlist(unname(rawdata[i,])), tigernorm[,i], batch = batch, type = metanorm.qc))
  dev.off()
}

In [ ]:
# ---- Dataset-BioHeart.R, lines 179-192 ----
# SERFF - manual process via SERFF webserver
# export data for SERFF
#SERFFdat <- rbind(batch, metanorm.qc, 1:ncol(rawdata), colnames(rawdata),rawdata)
#write.csv(SERFFdat, "Data/BioHeart/SERFFunnorm-full.csv")
# import normalized data
SERFFnorm <- read.csv("Data/BioHeart/normalizedbySERFF.csv", header = TRUE)[,-1]
SERFFnorm <- as.matrix(SERFFnorm)

plotdir = "Figures/BioHeart--metanorm-SERFF-rem2outliers/"
for(i in 1:nrow(SERFFnorm)){
  pdf(paste0(plotdir, "/", i, ".pdf"), width = 8, height = 5)
  print(plotSequence(unlist(unname(rawdata[i,-c(544, 1178)])), SERFFnorm[i,], batch = batch[-c(544, 1178)], type = metanorm.qc[-c(544, 1178)]))
  dev.off()
}

The panel itself is one `ggplot` per method over compound 36, all sharing `ylim(6, 16.5)` so the
eight sub-panels are directly comparable. Only the raw panel is reproduced here — the other seven differ
only in which normalized matrix supplies `y` and in the output filename.

In [ ]:
# ---- Dataset-BioHeart.R, lines 202-231 ----
colors <- c(
  "#4477AA", "#66CCEE", "#228833", "#CCBB44", "#EE6677",
  "#AA3377", "#BBBBBB", "#000000", "#77AADD", "#EE8866",
  "#44BB99", "#999933", "#AA4499", "#DDDDDD", "#332288"
)

# FIGURE 1
cmpID <- 36
dat <- data.frame(x = 1:ncol(rawdata),
                  y = unname(unlist(rawdata[cmpID,])),
                  type = metanorm.qc == "QC",
                  batch = batch)
dat$batch <- factor(dat$batch, levels = paste0("batch", 1:15))
dat$type[dat$type == TRUE] <- "QC"
dat$type[dat$type == "FALSE"] <- "sample"
preNorm <- ggplot(dat, aes(x = x,
                           y = y,
                           col = batch,
                           shape = type))+
  geom_point(size = 1.2, alpha = 0.8)+
  scale_color_manual(values = colors)+
  theme_custom()+
  xlab("injection order")+
  ylim(6, 16.5)+
  ylab("intensity")+
  theme(
    legend.key.height = unit(5, "points"),
    legend.spacing.y = unit(0, "cm"))
ggsave("Figures/ForPublication/Figure1/prepost-raw.pdf",
       device = cairo_pdf, width = 5, height = 1.3, units = "in")

### 1b — simulated drift, with and without outliers

The simulation draws a sine curve offset by 20, adds Gaussian noise (sd 0.3), splits the series into two
batches, and injects 0–4 outliers away from the sequence edges. Each of 200 runs fits every method to one
replicate and evaluates on the held-out replicate, so the error is a genuine prediction error rather than
a residual. `noutlier` at the top is the knob that produces the x-axis.

In [ ]:
# ---- simStudy-withOutliers.R, lines 200-202 ----

nsim <- 200
noutlier <- 1

In [ ]:
# ---- simStudy-withOutliers.R, lines 264-320 ----
for(coln in 1:nsim){
  cat(coln, " of ", nsim, "\n")
  
  ####
  # FIT FOR ONE REPLICATE, EVALUATE USING THE OTHER 
  
  loess.temp <- vector("numeric")
  robloess.temp <- vector("numeric")
  spline1.temp <- vector("numeric")
  gam1.temp <- vector("numeric")
  gam2.temp <- vector("numeric")

  loess.bias.temp <- vector("numeric")
  robloess.bias.temp <- vector("numeric")
  spline1.bias.temp <- vector("numeric")
  gam1.bias.temp <- vector("numeric")
  gam2.bias.temp <- vector("numeric")

  for(batch in 1:2){
    set.seed(coln)
    n <- 1000
    x <- seq(0, 5, length.out = n)
    true.fun <- sin(x) 
    true.fun <- true.fun[sort(c(seq(1, n, 12), seq(2, n, 12)))]
    true.fun <- true.fun + 20 # make sure no negative values, or close to 0
    x <- sort(c(seq(1, n, 12), seq(2, n, 12)))
    sd.noise <- 0.3
    y.sim <- true.fun + rnorm(length(x), mean = 0, sd = sd.noise)
    dat.full <- data.frame(x=x, y =y.sim, batch=factor(rep(c("B1", "B2"), each = length(x)/2)))
    dat <- dat.full[dat.full$batch == levels(dat.full$batch)[batch],]
    dat.repone <- dat[seq(1, nrow(dat), 2),]
    dat.reptwo <- dat[seq(2, nrow(dat), 2),]
    
    # add outlier(s) in each batch, replicate one
    #   not at the start or the end of the sequence (first/last 5 obs)
    idx <- sample(5:(nrow(dat.repone)-5), noutlier)
    dat.repone$y[idx] <- 
      dat.repone$y[idx] - 0.5*runif(noutlier, 0.4, 0.6)
    
    # standard loess fit, with span obtained by crossvalidation
    spanval <- selectSpan(dat.repone$x, dat.repone$y)
    fit <- loess(y~x, data = dat.repone, degree = 2, family = "gaussian", span = spanval)
    # robust loess fit, with span obtained by crossvalidation
    fit3 <- loess.as(dat.repone$x, dat.repone$y, degree = 2, criterion = "gcv", family = "symmetric")
 
    fit4 <- smooth.spline(dat.repone$x, dat.repone$y, cv = TRUE) 
    
    # gam fit
    fit5 <- gam(y ~ s(x, k = nrow(dat.repone)), data = dat.repone, method = "REML", family = scat(), control=list(maxit = 100))
    fit5r <- gam(y ~ s(x, k = nrow(dat.repone)), data = dat.repone, method = "REML")
    est.weights <- calcWeights(fit5r$residuals)
    fit5r <- gam(y ~ s(x, k = nrow(dat.repone)), data = dat.repone, method = "REML", weights = est.weights)
 
    loess.temp <- na.omit(c(loess.temp, dat.reptwo$y/predict(fit, newdata = data.frame(x = dat.reptwo$x))))
    robloess.temp <- na.omit(c(robloess.temp, dat.reptwo$y/predict(fit3, newdata = data.frame(x = dat.reptwo$x))))
    spline1.temp <- na.omit(c(spline1.temp, dat.reptwo$y/predict(fit4, newdata = data.frame(x = dat.reptwo$x))$y))
    gam1.temp <- na.omit(c(gam1.temp, dat.reptwo$y/predict(fit5, newdata = data.frame(x = dat.reptwo$x))))

The plotting script reads a summary CSV of the simulation results (European decimal format, hence
`sep = ";"`, `dec = ","`). Note there is no legend: the series are labelled with `geom_label()` at
hard-coded y positions just past the right edge of the panel, which is also why `plot.margin` reserves
90 pt on the right and `coord_cartesian(clip = "off")` is set. Those y values will need adjusting if you
re-run the simulation and the curves move.

In [ ]:
# ---- FigureMSPE.R, lines 31-61 ----

MSPEdf <- read.csv("Data/InSilico/MSPEsummary.csv", sep = ";", dec = ",")
head(MSPEdf)
library(ggplot2)
MSPEdf$QCs[MSPEdf$QCs == 1] <- "1 QC per 10 samples"
MSPEdf$QCs[MSPEdf$QCs == "2"] <- "2 QCs per 10 samples"
MSPEdf$QCs <- as.factor(MSPEdf$QCs)
p1 <- ggplot(MSPEdf, aes(x = Outliers, y = MSPE, color = Method, linetype = QCs))+
  geom_line(linewidth = 0.4) + geom_point(size = 1) +
  theme_custom(base_size = 11)+
  guides(color = "none")+
  theme(legend.position = "none")+
  scale_linetype_discrete(name = NULL) +
  ylab("relative MSPE (%)")+
  scale_x_continuous(limits = c(0, 4.10))+
  scale_color_brewer(palette = "Set1")+

  coord_cartesian(clip = "off")+
  xlab("outliers")+
  geom_label(x=4.13, y=263, label = "QC-RLSC (LOOCV)", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=243, label = "QC-RSC (LOOCV)", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=161, label = "rGAM", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=140, label = "tGAM", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=127, label = "rLOESS", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=80, label = "QC-RLSC (LOOCV)", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=92, label = "QC-RSC (LOOCV)", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=51, label = "rGAM", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=38, label = "tGAM", hjust=0, color = "black", family = "Arial", size = 3)+
  geom_label(x=4.13, y=64, label = "rLOESS", hjust=0, color = "black", family = "Arial", size = 3)
ggsave("Figures/aaForPublication/Figure1/Figure1B.pdf",
       device = cairo_pdf, width = 105, height = 110, units = "mm")

---

## Figure 2 — weighting biological samples, and knowing when to ignore the QCs

Four stacked panels for a single metabolite, Cyclo(Leucylprolyl), in the ENVIRONAGE urine study. Dark
symbols are QCs, light symbols biological samples; symbol shape encodes batch.

- **(a)** raw intensities, showing the drift and batch structure to be removed.
- **(b)** QC-RLSC: the fit follows outlying QC values, and the correction inherits that error.
- **(c)** tGAM with QC samples weighted 5:1 over biological samples, but with the QC check switched off.
  Drift is handled, yet a systematic offset remains, because the QCs sit at a different level from the
  biological samples — normalizing the samples by a QC-derived curve moves them to the wrong place.
- **(d)** the same tGAM fit with `QCcheck = TRUE`. A likelihood-ratio test compares the QC and sample
  intercepts and smooths within the batch; when they differ significantly the fit reverts to biological
  samples only, and the offset disappears.

The sequence (b) → (c) → (d) is the argument for the two refinements: differential weighting lets the
larger pool of biological samples contribute drift information, and the test decides per compound and per
batch whether the QCs deserve to drive the correction at all.

**Code note:** the repository has no script that produces this figure. The panels correspond to argument
settings on `metanorm()`; the cell below reconstructs them from the ENVIRONAGE data prepared in the next
section, following the description in the Experimental Section. Treat it as a sketch, not as the authors'
code — check `?metanorm` for the exact `weights` interface in the version you install.

In [ ]:
# Reconstruction (not from the repository) — panels b, c, d of Figure 2
# Panel b: QC-only, non-robust LOESS
qcrlsc <- metanorm(rawdata, type = metanorm.qc, batch = batch,
                   QConly = TRUE, model = "QC-RLSC", cv = "LOOCV")

# Panel c: tGAM, QCs weighted 5x relative to biological samples, no QC-vs-sample test
w <- ifelse(metanorm.qc == "QC", 5, 1)
tgam_noCheck <- metanorm(rawdata, type = metanorm.qc, batch = batch,
                         model = "tGAM", weights = w, QCcheck = FALSE)

# Panel d: same, with the representativeness test enabled
tgam_check <- metanorm(rawdata, type = metanorm.qc, batch = batch,
                       model = "tGAM", weights = w,
                       QCcheck = TRUE, QCcheckp = 0.05)

# Then draw each with the shared helper, for the compound of interest:
# plotSequence(unlist(rawdata[cmpID, ]), tgam_check[cmpID, ], batch = batch, type = metanorm.qc)

---

## Figure 3 — BioHEART: does the choice of method change the biology?

Five panels, moving from one metabolite to the whole study.

- **(a)** Methionine against injection order, raw / SERRF / tGAM. The discrepancy that motivated the
  panel: SERRF shows inflated variance among biological samples that sit near an outlying QC, while tGAM
  does not.
- **(b)** Bar chart of significant differentially abundant metabolites (hypertensive vs. not, Wilcoxon,
  p < 0.05) per method. Read it as a warning rather than a score — the *unnormalized* data yield the most
  DAMs, which is precisely why DAM count is a bad proxy for normalization quality.
- **(c)** Five-set Venn of which metabolites each method calls significant: tGAM, rGAM, rLOESS, SERRF and
  QC-RLSC. The robust three agree on about 92% of calls; SERRF overlaps them on only 29–41%.
- **(d)** PCA score plots (raw / SERRF / tGAM) coloured by batch. All normalization collapses the batch
  separation seen in the raw data; residual structure is still visible for SERRF and not for tGAM.
- **(e)** Heatmap of Spearman correlations between the methods' p-value vectors, rows and columns ordered
  by complete-linkage hierarchical clustering. This is the fine-grained version of the Venn: it does not
  threshold, so methods that merely shift p-values slightly stay close, and genuinely divergent methods
  split off as their own cluster.

Panel (b) counts and panel (c) membership both come from one p-value matrix with 11 columns — one per
method, including both cross-validation variants — so the GCV-vs-LOOCV comparison falls out of the same
object.

### 3a — one metabolite, three methods

In [ ]:
# ---- Dataset-BioHeart.R, lines 397-463 ----
# FIGURE 3A
dat <- data.frame(x = 1:ncol(rawdata),
                  y = unname(unlist(rawdata[50,])),
                  type = metanorm.qc == "QC",
                  batch = batch)
dat$batch <- factor(dat$batch, levels = paste0("batch", 1:15))
dat$type[dat$type == TRUE] <- "QC"
dat$type[dat$type == "FALSE"] <- "sample"
preNorm <- ggplot(dat, aes(x = x,
                           y = y,
                           col = batch,
                           shape = type))+
  geom_point(size = 1.2, alpha = 0.8)+
  scale_color_manual(values = colors)+
  theme_custom()+
  xlab("injection order")+
  ylab("intensity")+
  theme(
        legend.key.height = unit(5, "points"),
        legend.spacing.y = unit(0, "cm"))
ggsave("Figures/ForPublication/Figure3/prepost-raw2.pdf",
       device = cairo_pdf, width = 5, height = 1.3, units = "in")

dat <- data.frame(x = 1:ncol(rawdata),
                  y = unname(unlist(GAMnorm[50,])),
                  type = metanorm.qc == "QC",
                  batch = batch)
dat$batch <- factor(dat$batch, levels = paste0("batch", 1:15))
dat$type[dat$type == TRUE] <- "QC"
dat$type[dat$type == "FALSE"] <- "sample"
preNorm <- ggplot(dat, aes(x = x,
                           y = y,
                           col = batch,
                           shape = type))+
  geom_point(size = 1.2, alpha = 0.8)+
  scale_color_manual(values = colors)+
  theme_custom()+
  xlab("injection order")+
  ylab("intensity")+
  theme(
    legend.key.height = unit(5, "points"),
    legend.spacing.y = unit(0, "cm"))
ggsave("Figures/ForPublication/Figure3/prepost-tGAM2.pdf",
       device = cairo_pdf, width = 5, height = 1.3, units = "in")
dat <- data.frame(x = 1:ncol(rawdata),
                  y = unname(unlist(SERFFnorm[50,])),
                  type = metanorm.qc == "QC",
                  batch = batch)
dat$batch <- factor(dat$batch, levels = paste0("batch", 1:15))
dat$type[dat$type == TRUE] <- "QC"
dat$type[dat$type == "FALSE"] <- "sample"
preNorm <- ggplot(dat, aes(x = x,
                           y = y,
                           col = batch,
                           shape = type))+
  geom_point(size = 1.2, alpha = 0.8)+
  scale_color_manual(values = colors)+
  theme_custom()+
  xlab("injection order")+
  ylab("intensity")+
  theme(
    legend.key.height = unit(5, "points"),
    legend.spacing.y = unit(0, "cm"))
ggsave("Figures/ForPublication/Figure3/prepost-SERFF.pdf",
       device = cairo_pdf, width = 5, height = 1.3, units = "in")
ggsave("Figures/ForPublication/Figure3/prepost-forLegend.pdf",
       device = cairo_pdf, width = 5, height = 3.3, units = "in")

### 3d — PCA score plots

Runs with any missing value in *any* of the ten matrices are dropped first, so every panel is built on an
identical sample set. `plotPCA2` scales before `prcomp` and fixes the batch factor levels to `batch1`…
`batch15` so colours stay consistent across panels.

In [ ]:
# ---- Dataset-BioHeart.R, lines 470-495 ----
# remove incomplete cases for generating PCA plots
removeCases <- c(which(!complete.cases(t(rawdata))),
                 which(!complete.cases(t(GAMnorm))),
                 which(!complete.cases((tigernorm))),
                 which(!complete.cases(t(SERFFnorm))),
                 which(!complete.cases(t(QCRLSCGCVnorm))),
                 which(!complete.cases(t(QCRLSCLOOCVnorm))),
                 which(!complete.cases(t(QCRSCGCVnorm))),
                 which(!complete.cases(t(QCRSCLOOCVnorm))),
                 which(!complete.cases(t(rLOESSGCVnorm))),
                 which(!complete.cases(t(rLOESSLOOCVnorm))))
removeCases <- unique(removeCases)


extrafont::loadfonts()
p <- plotPCA2(GAMnorm[,-removeCases], type = batch[-removeCases])
ggsave("Figures/ForPublication/Figure3/PCA-tGAM.pdf",
       device = cairo_pdf, width = 3.5, height = 3, units = "in")
p <- plotPCA2(rawdata[,-removeCases], type = batch[-removeCases])
ggsave("Figures/ForPublication/Figure3/PCA-raw.pdf",
      device = cairo_pdf, width = 3.5, height = 3, units = "in")
p <- plotPCA2(SERFFnorm[,-removeCases], type = batch[-removeCases])
ggsave("Figures/ForPublication/Figure3/PCA-SERFF.pdf",
       device = cairo_pdf, family = "Arial", width = 3.5, height = 3, units = "in")
ggsave("Figures/ForPublication/Figure3/PCA-forLegend.pdf",
       device = cairo_pdf, family = "Arial", width = 3.5, height = 2.5, units = "in")

### 3b, 3c, 3e — differential abundance, Venn, and the p-value heatmap

The metadata join is worth noting: sample IDs are recovered from the column names of the original CSV
and matched to the hypertension flag, then written into row 3 of a copy of the raw table, which is what
the Wilcoxon loop reads.

In [ ]:
# ---- Dataset-BioHeart.R, lines 503-543 ----
# load the clinical metadata and match with the intensity data order
binaryCond <- read.csv("BioHEART_clinical_metadata.csv")
colIdLink <- data.frame(order = 1:ncol(metab), id = as.numeric(substr(colnames(metab), 2, 4)))
colIdLink <- left_join(colIdLink, binaryCond[, c("Pat.ID", "HTN")],
                       by = c("id" = "Pat.ID"))
metab2 <- metab
metab2[3,] <- colIdLink$HTN

# run Wilcoxon tests by method, store in a p value matrix, one column per method
pval.matrix <- matrix(1, nrow = length(1:nrow(rawdata)), ncol = 11)
for(metid in 1:nrow(rawdata)){
  pval.matrix[metid, 1] <- wilcox.test(as.numeric(unlist(rawdata[metid, 1:ncol(rawdata)]))~
                as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 2] <- wilcox.test(as.numeric(unlist(GAMnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 3] <- wilcox.test(as.numeric(unlist(t(tigernorm[,1:53])[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 4] <- wilcox.test(as.numeric(unlist(SERFFnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 5] <- wilcox.test(as.numeric(unlist(rGAMnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 6] <- wilcox.test(as.numeric(unlist(QCRLSCGCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 7] <- wilcox.test(as.numeric(unlist(QCRLSCLOOCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 8] <- wilcox.test(as.numeric(unlist(QCRSCGCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 9] <- wilcox.test(as.numeric(unlist(QCRSCLOOCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 10] <- wilcox.test(as.numeric(unlist(rLOESSGCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
  pval.matrix[metid, 11] <- wilcox.test(as.numeric(unlist(rLOESSLOOCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(unlist(metab2[3, 2:ncol(metab2)])))$p.value
}

# add col/row names, and generate a barplot for number of DAMs by method
colnames(pval.matrix) <- c("raw", "tGAM", "TIGER", "SERFF", "rGAM", "QCRLSC-GCV",
                           "QCRLSC-LOOCV", "QCRSC-GCV", "QCRSC-LOOCV", "rLOESS-GCV",
                           "rLOESS-LOOCV")
rownames(pval.matrix) <- metab2[4:nrow(metab2), 1]
cormat <- cor(pval.matrix, method="spearman")

In [ ]:
# ---- Dataset-BioHeart.R, lines 545-553 ----
signifDf <- data.frame(count = colSums(pval.matrix < 0.05))
signifDf$method <- rownames(signifDf)

p1 <- ggplot(signifDf, aes(x=method, y=count))+
  geom_bar(stat="identity") +
  theme_custom()+
  ylab("DAMs") + xlab("")+coord_flip()
ggsave("Figures/ForPublication/Figure3/barplot.pdf",
       device = cairo_pdf, family = "Arial", width = 3, height = 3.5, units = "in")

In [ ]:
# ---- Dataset-BioHeart.R, lines 556-579 ----
# generate a 5 group Venn diagram, compare robust to state of the art methods
signifIds <- apply((pval.matrix[,c(2, 5, 4, 7, 10)]), 2, function(x){which(x<0.05)})
display_venn(
  signifIds,
  category.names = c("tGAM" , "rGAM" , "SERFF", "QC-RLSC", "rLOESS"),
  fill = c("#4D7C8A", "#8FAD88", "#CBDF90", "#EDBFB7", "#F4A261"),
  cat.dist = c(0.1, 0.1, 0.1, 0.1, 0.1),
  fontfamily = "sans",
  cat.fontfamily = "sans",
  cat.fontface = "bold"
)
pdf("Figures/ForPublication/Figure1/venn.pdf",
       width = 6, height = 5)
display_venn(
  signifIds,
  category.names = c("" , "" , "", "", ""),
  fill = c("#4D7C8A", "#8FAD88", "#CBDF90", "#EDBFB7", "#F4A261"),
  cat.dist = c(0.1, 0.1, 0.1, 0.1, 0.1),
  fontfamily = "sans",
  cat.fontfamily = "sans",
  cat.fontface = "bold",
  cex = 1.2
)
dev.off()

In [ ]:
# ---- Dataset-BioHeart.R, lines 583-605 ----
# generate method correlation heatmap
d <- as.dist(1 - cormat)
hc <- hclust(d)

cormat_reordered <- cormat[hc$order, hc$order]

cormatdf <- melt(cormat_reordered)
colnames(cormatdf) <- c("row", "col", "correlation")

cormatdf$row <- factor(cormatdf$row, levels = rev(rownames(cormat_reordered)))
cormatdf$col <- factor(cormatdf$col, levels = colnames(cormat_reordered))

p <- ggplot(cormatdf, aes(x = col, y = row, fill = correlation)) +
  geom_tile() +
  scale_fill_gradient2(low = "blue2", mid = "white", high = "darkorange2", 
                       midpoint = 0) +
  theme_custom() +
  theme(axis.title.x = element_blank(),
        axis.title.y = element_blank(),
        axis.text.x = element_text(angle = 45, hjust = 1)) +
  coord_fixed()
ggsave("Figures/ForPublication/Figure3/heatmap.pdf",
       device = cairo_pdf, width = 3.5, height = 3, units = "in")

---

## Figure 4 — replicate runs as an outcome-free benchmark

A single panel: one bar (or point) per normalization method, showing the median absolute deviation of
scaled differences between two runs of the same biological sample analysed in different BioHEART batches.

The logic is what makes this the most trustworthy panel in the paper. Because normalization was done
batchwise, a replicate in batch *k* carries no information into batch *l*, so the two corrected
intensities are independent estimates of the same quantity. Any disagreement between them is technical
variance that normalization failed to remove — no biological outcome, no p-value threshold, no
QC-based circularity. **Lower is better.** The reported reductions relative to raw data are 49–52% for the
robust methods, 41% for SERRF, 40% for TIGER, and 18–22% for QC-RLSC / QC-RSC.

**Code note:** no script in `Rcode/` computes this. The cell below implements the recipe given in the
Experimental Section — per replicate pair, take the difference of the two intensities rescaled by their
mean, then the MAD of those differences across all compounds and pairs. You will need the replicate
pairing, which lives in the BioHEART metadata rather than in the intensity matrix.

In [ ]:
# Reconstruction (not from the repository) — Figure 4
# `pairs` must be a two-column matrix of column indices into the intensity matrices,
# each row a biological sample run twice in different batches.

replicateMAD <- function(mat, pairs) {
  d <- apply(pairs, 1, function(p) {
    a <- as.numeric(mat[, p[1]])
    b <- as.numeric(mat[, p[2]])
    (a - b) / ((a + b) / 2)          # difference rescaled by the pair mean
  })
  mad(as.numeric(d), na.rm = TRUE)   # lower = better cross-batch agreement
}

mats <- list(raw = rawdata, tGAM = GAMnorm, rGAM = rGAMnorm,
             rLOESS = rLOESSGCVnorm, SERFF = SERFFnorm, TIGER = t(tigernorm),
             `QC-RLSC` = QCRLSCLOOCVnorm, `QC-RSC` = QCRSCLOOCVnorm)

madDf <- data.frame(method = names(mats),
                    mad = sapply(mats, replicateMAD, pairs = pairs))

ggplot(madDf, aes(x = reorder(method, mad), y = mad)) +
  geom_bar(stat = "identity") + coord_flip() +
  theme_custom() + xlab("") + ylab("MAD of scaled between-batch differences")

---

## Figure 5 — ENVIRONAGE: the same comparison, milder conditions

Figure 5 repeats Figure 3's five panels on the ENVIRONAGE urine cohort (393 runs, 6 batches, 269 targeted
metabolites), with Phenylglycine as the example metabolite in panel (a) and biological sex as the grouping
for the DAM analysis in (b), (c) and (e). Panel (d) is again PCA by batch.

The differences from Figure 3 are the interesting part. Drift and batch effects are smaller here and the
outliers are fewer and milder, so the methods converge: the robust three overlap on 74–88% of DAMs, TIGER
and SERRF on 64–70%, and QC-RLSC on 47–51%. Most remaining disagreements are small p-value shifts around
the 0.05 line — visible as high off-diagonal correlations in the heatmap — rather than substantive
divergence. The exceptions trace back to two now-familiar causes: QC-RLSC fitting unrepresentative QCs,
and SERRF reacting to outliers.

One dataset-specific caveat carried in the code: some batches do not start with a QC injection, and
QC-RLSC / QC-RSC cannot extrapolate before the first QC in a batch, so those leading samples are missing
for both methods.

### Data preparation

More involved than BioHEART: negative and positive polarity tables are concatenated, compounds with more
than 40 missing values (>10%) dropped, log-transformed, and then reordered — the columns do not arrive in
analysis order, so the run sequence is read from a separate spreadsheet and used to sort. Batches are
assigned from lab records as run-length blocks. Compounds 77 and 245 are removed because QC-RLSC fails on
them (NAs in QCs), keeping the method comparison on a common feature set.

In [ ]:
# ---- Dataset-ENVIRONAGE.R, lines 126-182 ----
# read and combine negative and positive polarity results
metab <- read.csv("Data/ENVIRONAGE/auc_table_negative.csv", header = TRUE)
metab2 <- read.csv("Data/ENVIRONAGE/auc_table_positive.csv", header = TRUE)
metab <- rbind(metab, metab2)

# remove compounds with > 10% = 40 missing values
# clean up, and log transform
metab <- metab[rowSums(is.na(metab)) < 40,]
metab <- metab[, - c(1:2)]
metab <- metab[, - c(394:395)]
metab <- log(metab)
metab <- as.matrix(metab)

# reorder the analyses because they are not in the original analysis order
runseq <- read_xlsx("Data/ENVIRONAGE/run_seq.xlsx")
metab.order <- substr((colnames(metab)), 2, 20)
metab.order <- unlist(strsplit(metab.order, "\\."))[seq(1, 2*ncol(metab), 2)]
getOrder <- numeric(length(metab.order))
for(file in 1:length(getOrder)){
  getOrder[file] <- ifelse(length(which(metab.order[file] == runseq$file)) == 0,
                           NA,
                           which(metab.order[file] == runseq$file))
}
metab <- metab[,order(getOrder)]

# get QC locations and define the batches based on lab information
metanorm.qc <- rep("sample", 393)
metanorm.qc[grep("QC", colnames(metab))] <- "QC"

batch <- c(rep("batch1", 58),
           rep("batch2", 72),
           rep("batch3", 60),
           rep("batch4", 71),
           rep("batch5", 60),
           rep("batch6", 72))

# get the study ID which is needed for DAM analysis later on
studyId <- unlist(strsplit(metab.order, "\\_"))[seq(3, length(metab.order)*3, 3)]
studyId <- unlist(strsplit(colnames(metab), "\\_"))[seq(3, length(colnames(metab))*3, 3)]
studyId <- unlist(strsplit(studyId, "\\."))[seq(1, length(colnames(metab))*2, 2)]
studyId[metanorm.qc == "QC"] <- paste0("QC", studyId[metanorm.qc == "QC"])

# get the final raw data for normalization
rawdata <- metab
# failure due to NAs in QCs for comps 77 and 245 for QC-RLSC, 
#   remove for honest comparison between methods
rawdata <- rawdata[-c(77,245),]

# normalizatin using the metanorm package
GAMnorm <- metanorm(rawdata, type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/ENVIRONAGEbatchw-metanorm-2/")
rGAMnorm <- metanorm(rawdata, model="rGAM", type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/ENVIRONAGEbatchwrgam-metanorm-2/")
QCRLSCLOOCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "LOOCV", plotdir = "Figures/ENVIRONAGE-metanorm-QCRLSC-LOOCV/")
QCRLSCGCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "GCV", plotdir = "Figures/ENVIRONAGE-metanorm-QCRLSC-GCV/")
QCRSCLOOCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "LOOCV", plotdir = "Figures/ENVIRONAGE-metanorm-QCRSC-LOOCV/")
QCRSCGCVnorm <- metanorm(rawdata, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "GCV", plotdir = "Figures/ENVIRONAGE-metanorm-QCRSC-GCV/")
rLOESSLOOCVnorm <- metanorm(rawdata, type = metanorm.qc, batch = batch, model = "rLOESS", cv = "LOOCV", plotdir = "Figures/ENVIRONAGE-metanorm-rLOESS-LOOCV/")
rLOESSGCVnorm <- metanorm(rawdata, type = metanorm.qc, batch = batch, model = "rLOESS", cv = "GCV", plotdir = "Figures/ENVIRONAGE-metanorm-rLOESS-GCV/")

### Panels

Panel (a) at lines 346–454, PCA at 455–476, DAM barplot at 477 onward; the Wilcoxon loop and Venn mirror
the BioHEART script with `colIdLink$Group` (sex) as the grouping variable. The heatmap block is shown
here as the representative excerpt.

In [ ]:
# ---- Dataset-ENVIRONAGE.R, lines 242-280 ----
# do the DAM analysis using individuals' biological sex
# first match sex to samples using their IDs
binaryCond <- read_xlsx("Metadata ENVIRONAGE_Birth.xlsx", sheet = 1)
binaryCond$Sex <- as.factor(binaryCond$Sex)
levels(binaryCond$Sex) <- c("male", "female")
binaryCond$SampleName <- as.character(binaryCond$SampleName)
colIdLink <- data.frame(order = 1:ncol(rawdata), id = studyId)
colIdLink <- left_join(colIdLink, binaryCond[, c("SampleName", "Sex")],
                       by = c("id" = "SampleName"))
colIdLink$Group <- colIdLink$Sex

# WIlcoxon tests for DAM analysis
pval.matrix <- matrix(1, nrow = length(1:nrow(rawdata)), ncol = 11)
for(metid in 1:nrow(rawdata)){
  pval.matrix[metid, 1] <- wilcox.test(as.numeric(unlist(rawdata[metid, 1:ncol(rawdata)]))~
                as.factor(colIdLink$Group))$p.value

  pval.matrix[metid, 2] <- wilcox.test(as.numeric(unlist(GAMnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 3] <- wilcox.test(as.numeric(unlist(t(tigernorm[,1:269])[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 4] <- wilcox.test(as.numeric(unlist(SERFFnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 5] <- wilcox.test(as.numeric(unlist(rGAMnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 6] <- wilcox.test(as.numeric(unlist(QCRLSCGCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 7] <- wilcox.test(as.numeric(unlist(QCRLSCLOOCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 8] <- wilcox.test(as.numeric(unlist(QCRSCGCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 9] <- wilcox.test(as.numeric(unlist(QCRSCLOOCVnorm[metid, 1:ncol(rawdata)]))~
                                         as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 10] <- wilcox.test(as.numeric(unlist(rLOESSGCVnorm[metid, 1:ncol(rawdata)]))~
                                          as.factor(colIdLink$Group))$p.value
  pval.matrix[metid, 11] <- wilcox.test(as.numeric(unlist(rLOESSLOOCVnorm[metid, 1:ncol(rawdata)]))~
                                          as.factor(colIdLink$Group))$p.value
}

In [ ]:
# ---- Dataset-ENVIRONAGE.R, lines 316-345 ----
# generate heatmaps
d <- as.dist(1-cormat)
hc <- hclust(d)

cormat_reordered <- cormat[hc$order, hc$order]

cormatdf <- melt(cormat_reordered)
colnames(cormatdf) <- c("row", "col", "correlation")

cormatdf$row <- factor(cormatdf$row, levels = rev(rownames(cormat_reordered)))
cormatdf$col <- factor(cormatdf$col, levels = colnames(cormat_reordered))

p <- ggplot(cormatdf, aes(x = col, y = row, fill = correlation)) +
  geom_tile() +
  scale_fill_gradient2(low = "blue2", mid = "white", high = "darkorange2", midpoint = 0) +
  theme_custom() +
  theme(axis.title.x = element_blank(),
        axis.title.y = element_blank(),
        axis.text.x = element_text(angle = 45, hjust = 1)) +
  coord_fixed()
ggsave("Figures/ForPublication/Figure4/heatmap.pdf",
       device = cairo_pdf, width = 3.5, height = 3, units = "in")

---

## Figure 6 — what robustness costs in compute

Two panels, both with cores (1–6) on the x-axis and **compounds normalized per minute** on the y-axis —
throughput, so higher is better. Panel (a) is BioHEART (53 targeted compounds, 1361 runs, 15 batches);
panel (b) is FAME (8913 features, 618 runs, 9 batches).

Two readings. Vertically: QC-RSC is fastest, with rLOESS and QC-RLSC under GCV close behind; the GAM
methods and TIGER cost substantially more. Horizontally: the *slope* shows whether parallelization pays.
On BioHEART the fast methods flatten and then decline as cores are added — with only 53 compounds the
per-core overhead outweighs the work — while on FAME every method keeps improving, implying that more
than 6 cores would still help on untargeted data. The method ordering differs between panels because the
two datasets differ in both run count and batch count, which load the methods differently.

SERRF appears in neither panel: it runs only through its web interface, took 147 s on BioHEART, and
failed outright on FAME.

The timings themselves come from `methodComparisonTimeBioHeart.R` and `methodComparisonTimeFAME.R`,
which are deliberately repetitive: the whole block of `metanorm()` calls is repeated once per core count,
wrapped in timing calls, with a pause between runs so CPU throttling does not contaminate the next
measurement. Results are saved to `.RData` and plotted by `FigureTime.R`. As with Figure 1b, the series
labels are hard-coded `geom_label()` positions rather than a legend.

In [ ]:
# ---- methodComparisonTimeBioHeart.R, lines 1-45 ----
library(metanorm)
library(TIGERr)
library(ggplot2)
library(cowplot)
library(dplyr)
library(VennDiagram)

metab <- read.csv("~/Downloads/BioHEART_metabolomics_filtered.csv", header = TRUE)
metanorm.qc <- unname(unlist(metab[2, 2:ncol(metab)]))
metanorm.qc[metanorm.qc == "QC"] <- "QC"
metanorm.qc[metanorm.qc != "QC"] <- "sample"
batch <- paste0("batch", unlist(metab[3, 2:ncol(metab)]))
rawdata <- log(read.csv("~/Downloads/BioHEART_metabolomics_filtered.csv", header = TRUE, skip = 3)[,-1])

ncpu = 6
time <- Sys.time()
GAMnorm <- metanorm(rawdata[,], ncpu = ncpu, type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/BioHeartbatchw-metanorm/")
GAM.elapsed <- Sys.time() - time
Sys.sleep(120)
time <- Sys.time()
rGAMnorm <- metanorm(rawdata[,], ncpu = ncpu,  model="rGAM", type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/BioHeartbatchwrgam-metanorm/")
rGAM.elapsed <- Sys.time() - time
Sys.sleep(120)
time <- Sys.time()
QCRLSCLOOCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "LOOCV", plotdir = "Figures/BioHeart-metanorm-QCRLSC-LOOCV/")
QCRLSCLOOCV.elapsed <- Sys.time() - time
Sys.sleep(120)
time <- Sys.time()
QCRLSCGCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "GCV", plotdir = "Figures/BioHeart-metanorm-QCRLSC-GCV/")
QCRLSCGCV.elapsed <- Sys.time() - time
Sys.sleep(120)
time <- Sys.time()
QCRSCLOOCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "LOOCV", plotdir = "Figures/BioHeart-metanorm-QCRSC-LOOCV/")
QCRSCLOOCV.elapsed <- Sys.time() - time
Sys.sleep(120)
time <- Sys.time()
QCRSCGCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "GCV", plotdir = "Figures/BioHeart-metanorm-QCRSC-GCV/")
QCRSCGCV.elapsed <- Sys.time() - time
Sys.sleep(120)
time <- Sys.time()
rLOESSGCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, batch = batch, model = "rLOESS", cv = "GCV", plotdir = "Figures/BioHeart-metanorm-rLOESS-GCV/")
rLOESSGCV.elapsed <- Sys.time() - time
Sys.sleep(120)

# TIGERr

In [ ]:
# ---- FigureTime.R, lines 28-69 ----
}


load("Data/618samplesandqcs-normalizationTime.RData")
p1 <- ggplot(timing, aes(x = cores, y = 1/(as.numeric(time)/1000), linetype = method))+
  geom_line()+
  geom_point(size = 1)+
  theme_custom(base_size=18)+
  theme(legend.position = "none")+
  xlab("cores")+
  ylab("compounds per minute")+
  scale_x_continuous(limits = c(1, 6))+
  coord_cartesian(clip = "off")+
  geom_label(x=6.15, y=58, label = "TIGER", hjust=0, size = 5)+
  geom_label(x=6.15, y=593, label = "tGAM", hjust=0, size = 5)+
  geom_label(x=6.16, y=763, label = "rGAM", hjust=0, size = 5)+
  geom_label(x=6.15, y=148, label = "QC-RLSC LOOCV", hjust=0, size = 5)+
  geom_label(x=6.15, y=1462, label = "QC-RLSC GCV", hjust=0, size = 5)+
  geom_label(x=6.15, y=1308, label = "rLOESS", hjust=0, size = 5)+
  geom_label(x=6.15, y=1698, label = "QC-RSC LOOCV", hjust=0, size = 5)
ggsave("Figures/ForPublication/Figure5/Figure5a.pdf",
       device = cairo_pdf, width = 110, height = 160, units = "mm")

load("Data/BioHeart/normalizationTime.RData")
p1 <- ggplot(timing, aes(x = cores, y = 1/(as.numeric(time)/1000), linetype = method))+
  geom_line()+
  geom_point(size = 1)+
  theme_custom(base_size=18)+
  theme(legend.position = "none")+
  xlab("cores")+
  ylab("compounds per minute")+
  scale_x_continuous(limits = c(1, 6))+
  coord_cartesian(clip = "off")+
  geom_label(x=6.15, y=15.7, label = "TIGER", hjust=0, size = 5)+
  geom_label(x=6.15, y=7.4, label = "tGAM", hjust=0, size = 5)+
  geom_label(x=6.15, y=57.0, label = "rGAM", hjust=0, size = 5)+
  geom_label(x=6.15, y=74.8, label = "QC-RLSC LOOCV", hjust=0, size = 5)+
  geom_label(x=6.15, y=116.5, label = "QC-RLSC GCV", hjust=0, size = 5)+
  geom_label(x=6.15, y=106.6, label = "rLOESS", hjust=0, size = 5)+
  geom_label(x=6.15, y=126.0, label = "QC-RSC LOOCV", hjust=0, size = 5)
ggsave("Figures/ForPublication/Figure5/Figure5b.pdf",
       device = cairo_pdf, width = 110, height = 160, units = "mm")

---

## Supplementary — FAME (untargeted)

`dataset-FAME.R` applies the same pipeline to 8913 saliva features across 618 runs, grouping by weight
status for the DAM analysis, and writes to `Figures/ForPublication/FigureSupplFAME/`. It differs from the
other two scripts in three ways: `ncpu` is passed to every `metanorm()` call, rows containing `Inf` after
log-transformation are dropped, and there is no SERRF arm at all — the dataset exceeded the web server's
capacity.

In [ ]:
# ---- dataset-FAME.R, lines 126-151 ----
load("618samplesandqcs.RData")

# extract the intensity data and metadata
rawdata <- log(t(sampleMetadata[,9:8921]))
colnames(rawdata) <- as.character(1:618)
metanorm.qc <- sampleMetadata$Type
metanorm.qc[metanorm.qc == "IQC"] <- "QC"
metanorm.qc[metanorm.qc != "QC"] <- "sample"
batch <- paste0("batch", substr(sampleMetadata$SampleName, 6, 7))
# remove rows with Inf vals
rawdata <- rawdata[-which(rowSums(is.infinite(rawdata))>0),]

#####################
# NORMALIZATION
#####################

GAMnorm <- metanorm(rawdata, ncpu = ncpu,  model="rGAM", type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/FAMEbatchwgam-metanorm/")
rGAMnorm <- metanorm(rawdata, ncpu = ncpu,  model="rGAM", type = metanorm.qc, QCcheck = TRUE, batch = batch, plotdir = "Figures/FAMEbatchwrgam-metanorm/")
QCRLSCGCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "GCV", plotdir = "Figures/FAME-metanorm-QCRLSC-GCV/")
QCRSCGCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "GCV", plotdir = "Figures/FAME-metanorm-QCRSC-GCV/")
rLOESSGCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, batch = batch, model = "rLOESS", cv = "GCV", plotdir = "Figures/FAME-metanorm-rLOESS-GCV/")
QCRLSCLOOCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RLSC", cv = "LOOCV", plotdir = "Figures/FAME-metanorm-QCRLSC-LOOCV/")
QCRSCLOOCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, QConly = TRUE, batch = batch, model = "QC-RSC", cv = "LOOCV", plotdir = "Figures/FAME-metanorm-QCRSC-LOOCV/")
rLOESSLOOCVnorm <- metanorm(rawdata,  ncpu = ncpu, type = metanorm.qc, batch = batch, model = "rLOESS", cv = "LOOCV", plotdir = "Figures/FAME-metanorm-rLOESS-LOOCV/")

---

## Data availability

| Dataset | Where |
|---|---|
| BioHEART (targeted plasma) | https://github.com/SydneyBioX/BioHEART_metabolomics |
| ENVIRONAGE (targeted urine) | https://zenodo.org/records/14548033 |
| FAME (untargeted saliva) | Metabolomics Workbench, study ST003697 (project PR002287) |

The scripts read these from paths like `Data/BioHeart/...`, `Data/ENVIRONAGE/...` and a couple of
absolute `~/Downloads/...` paths; adjust before running. SERRF-normalized matrices are not in the
repository — they are produced by uploading the exported CSV (the commented-out `write.csv` lines) to
https://slfan.shinyapps.io/ShinySERRF/ and downloading the result.

## Reproducibility notes

- R 4.4.1 on a 10-core Apple M1 Pro, 16 GB RAM; package versions in the paper's Supplementary Table S1.
- `theme_custom()` requires the Arial font via `extrafont`; on Linux either install it or change
  `base_family`, otherwise `cairo_pdf` output will fall back silently.
- Several `ggsave()` calls follow a plot assigned to a variable rather than printed, so they save the last
  plot built — keep the assignment and the save adjacent if you reorder cells.
- All `ggsave()` targets assume the `Figures/ForPublication/FigureN/` tree already exists; `ggsave` does
  not create directories.